In [2]:
import datetime as dt
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools


In [3]:

def extract_msg(line):
    "Regex to extract datetime, sender and message."
    date = []
    time = []
    msg = []
    sender_name = []
    datetime_pat  = "(\d+/\d+/\d+) (\d+:\d+) - (.*?): (.*)"  
    msgdata = re.findall(datetime_pat, line)
    date,time,sender_name, msg = msgdata[0][0],msgdata[0][1],msgdata[0][2],msgdata[0][3]
#     sender_pat = "\-\ [^:]*" #"\-\ \w+\:|\-\ \w+\ \w+\:" # one or two names
#     sender = re.findall(sender_pat,line)

#     if date and sender:
#         if len(sender.group(0)) < 25:
#             # assumes a name and last name is less than at most N chars. 
#             # Avoids misclassifying a status change with a semi-colon for a name.
#             date = pd.to_datetime(date.group(0))        
#             sender_name = sender.group(0)[2:]

#             msg = line[line.index(sender_name)+len(sender_name)+2:]
        
    return date, time, sender_name, msg

# parse the entire convo as a pd.dataframe

# f = open('chat/chat.txt', 'r', encoding='utf-8')
# conv0 = pd.DataFrame(columns=['date','sender','message'])
# for count, line in enumerate(f):
#     date, sender, msg  = extract_msg(line)
#     if sender and msg:
#         temp_df = pd.Series({'date':date ,'sender':sender,'message':msg})
#         conv0 = conv0.append(temp_df, ignore_index=True)
        
#f.close()

In [4]:
f = open('chat/chat.txt', 'r', encoding='utf-8')
content = f.readlines()
content = [x.strip() for x in content] 
conv0 = pd.DataFrame(columns=['date',"time",'sender','message'])
erros = []
for i in range(len(content)):
    try :
        date,time,sender, msg  = extract_msg(content[i]);
        temp_df = pd.Series({'date':date ,"time": time,'sender':sender,'message':msg}) 
        conv0 = conv0.append(temp_df, ignore_index=True)
    except IndexError:
        #print(i)
        pass

f.close()




In [5]:
conv0

,date,time,sender,message
0,11/6/18,21:49,Milena,roberto
1,11/6/18,21:49,Milena,posso te pedir um conselho
2,11/6/18,21:49,Milena,?
3,11/6/18,21:51,Roberto,Pode
4,11/6/18,21:51,Milena,eu desisto de cálculo?
...,...,...,...,...
70877,26/1/20,19:31,Roberto,Mi?
70878,26/1/20,19:31,Roberto,Td bemM
70879,26/1/20,19:31,Roberto,?
70880,26/1/20,20:56,Milena,Tudo bem e vc?


In [6]:
#a = conv0.groupby("date")
#len(a.get_group("1/10/19").groupby("sender").get_group("Milena").index)
#Organiza el dataframe por dia, despues por remitente y despues por remitente especifico (roberto) y
#cuenta el numero de miembro
conv1copia = conv0

In [7]:
#Generamos todos os dias
totaldedias = pd.date_range(start='2018-06-11', end='2020-01-24', periods=500).to_pydatetime().tolist()
diastot = []
for i in totaldedias:
    diastot.append(i.strftime("%d/%-m/%y")) #Lista dos totais

semduplicado = conv1copia.drop_duplicates("date")#Tirando duplicados
fechas = semduplicado["date"].tolist() #Lista dos usados

for dia in diastot: #Colocando todas as fechas no array
    if dia not in fechas:
        fechas.append(dia)
        
        

In [8]:
pd.to_datetime(conv0['time'], format='%H:%M')[0]

Timestamp('1900-01-01 21:49:00')

In [9]:
msgcadaum = pd.DataFrame(columns=['date','msgmi','msgro']) #msgs roberto e da milena

convpordate = conv0.groupby("date")

for i in fechas:
    try:
        msgmi = len(convpordate.get_group(i).groupby("sender").get_group("Milena").index) #Separa msg por dia(i) depois por autor e agarra especificos
        msgro = len(convpordate.get_group(i).groupby("sender").get_group("Roberto").index)
        tiempito = convpordate.get_group(i).groupby("sender").get_group("Roberto").index
        temp_df = pd.Series({'date':i ,"time": ,"msgmi": msgmi,'msgro':msgro }) 
        msgcadaum = msgcadaum.append(temp_df, ignore_index=True)
    except KeyError as e:
        #print ('I got a KeyError - reason "%s"' % str(e)); #Error significa sin msgs
        msgmi = 0
        msgro = 0
        temp_df = pd.Series({'date':i ,"time": ,"msgmi": msgmi,'msgro':msgro }) 
        msgcadaum = msgcadaum.append(temp_df, ignore_index=True)
    


#plt.plot(temp_df["date"], msgmi)
#msgcadaum["date"]

SyntaxError: invalid syntax (<ipython-input-9-660cc114e67d>, line 10)

In [ ]:
#Ordenamos msg por fecha e cambiamos el formato

msgcadaum['date'] =pd.to_datetime(msgcadaum.date, format='%d/%m/%y')
msgcadaum = msgcadaum.sort_values(by='date')
msgcadaum["date"] = msgcadaum["date"].apply(lambda x: x.strftime('%d/%m/%y')) 


In [ ]:
#Plot lineal

fig, ax1 = plt.subplots(figsize=(30,20))
ax1.set_ylabel(r"Número de mensajes enviados")
ax1.set_xlabel(r"Dia")
ax1.xaxis.set_major_locator(plt.MaxNLocator(130))
plt.xticks(rotation=45)



plt.plot(msgcadaum["date"], msgcadaum["msgmi"], label = "Milena")
plt.plot(msgcadaum["date"], msgcadaum["msgro"], label = "Roberto")
ax1.legend(loc='best')
fig.savefig("robemiPlot.pdf", bbox_inches='tight')




In [ ]:
#Plot scatter
fig, ax1 = plt.subplots(figsize=(30,20))
ax1.set_ylabel(r"Número de mensajes enviados")
ax1.set_xlabel(r"Dia")
ax1.xaxis.set_major_locator(plt.MaxNLocator(130))
plt.xticks(rotation=45)


plt.scatter(msgcadaum["date"], msgcadaum["msgmi"], label = "Milena")
plt.scatter(msgcadaum["date"], msgcadaum["msgro"], label = "Roberto")

ax1.legend(loc='best')

fig.savefig("robemiSca.pdf", bbox_inches='tight')



In [ ]:
#msg totais do roberto e da milena

totalmi = msgcadaum["msgmi"].sum()
totalro = msgcadaum["msgro"].sum()

totalmi,totalro

In [ ]:
msgcadaum

In [ ]:
msgcadaum['time'] =pd.to_datetime(msgcadaum.date, format='%H:%M:%S')


In [ ]:
msgcadaum['time']